In [ ]:
# run this notebook after 26_methods_ancestry_R
# use this notebook to compare mutation rates across parental smoking status and generate files for
# smoking spectrum inference
# we need ukb trio parent smoking status (./ukb/ukb_trios_smoking_status.csv)
# after this, run 28_methods_smoking_python

In [ ]:
library(jcolors)
library(data.table)
library(ggplot2)
library(dplyr)
library(jsonlite)
library(tidyverse)
library(ggpubr)
library(tidyr)

In [ ]:
counts = fread("./all_counts.txt") %>% filter(source == "trio")
ukb_smoking = fread("./ukb/ukb_trios_smoking_status.csv", sep = ",", header=TRUE)

In [ ]:
library(tidyverse)
library(bigrquery)

# this query represents dataset "smoking" for domain "survey" and was generated for all of us controlled tier dataset v8
dataset_61021453_survey_sql = paste("
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer,
        answer.survey_version_concept_id,
        answer.survey_version_name  
    FROM
        `ds_survey` answer   
    WHERE
        (
            question_concept_id IN (SELECT
                DISTINCT concept_id                         
            FROM
                `cb_criteria` c                         
            JOIN
                (SELECT
                    CAST(cr.id as string) AS id                               
                FROM
                    `cb_criteria` cr                               
                WHERE
                    concept_id IN (1585855)                               
                    AND domain_id = 'SURVEY') a 
                    ON (c.path like CONCAT('%', a.id, '.%'))                         
            WHERE
                domain_id = 'SURVEY'                         
                AND type = 'PPI'                         
                AND subtype = 'QUESTION')
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `cb_search_person` p 
                WHERE
                    has_whole_genome_variant = 1 ) )
        )", sep="")

# formulate a cloud storage destination path for the data exported from bigquery.
# note: by default data exported multiple times on the same day will overwrite older copies.
#       but data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
survey_61021453_path = file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
#  strftime(lubridate::now(), "%Y%m%d"),  # comment out this line if you want the export to always overwrite.
  "survey_61021453",
  "survey_61021453_*.csv")
message(str_glue('The data will be written to {survey_61021453_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

#perform the query and export the dataset to cloud storage as csv files.

bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_61021453_survey_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  survey_61021453_path,
  destination_format = "CSV")

read_bq_export_from_workspace_bucket = function(export_path) {
  col_types = cols(survey = col_character(), question = col_character(), answer = col_character(), survey_version_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk = read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types = spec(chunk)
          }
          chunk
        }))
}
dataset_61021453_survey_df = read_bq_export_from_workspace_bucket(survey_61021453_path)

dim(dataset_61021453_survey_df)

In [ ]:
aou_smoking = dataset_61021453_survey_df %>% filter(question == 'Smoking: 100 Cigs Lifetime') %>% select(person_id, answer)
names(aou_smoking) = c("idv", "smoking")
# read trios for aou 
aou_trios = fread("./relatedness/trio_counts.txt")
aou_trios$mother_id = ifelse(aou_trios$sex_par1 == "Female", aou_trios$par1, aou_trios$par2)
aou_trios$father_id = ifelse(aou_trios$sex_par1 == "Male", aou_trios$par1, aou_trios$par2)
ukb_smoking$Paternal = ifelse(ukb_smoking$Paternal %in% c("Previous", "Current"), "Yes", ukb_smoking$Paternal)
ukb_smoking$Paternal = ifelse(ukb_smoking$Paternal %in% c("Never"), "No", ukb_smoking$Paternal)
ukb_smoking$Paternal = ifelse(ukb_smoking$Paternal %in% c('Prefer not to answer'), NA, ukb_smoking$Paternal)
ukb_smoking$Maternal = ifelse(ukb_smoking$Maternal %in% c("Previous", "Current"), "Yes", ukb_smoking$Maternal)
ukb_smoking$Maternal = ifelse(ukb_smoking$Maternal %in% c("Never"), "No", ukb_smoking$Maternal)
ukb_smoking$Maternal = ifelse(ukb_smoking$Maternal %in% c('Prefer not to answer'), NA, ukb_smoking$Maternal)
family = fread("./aou_ukb_families_COSMIC_96_all_samples.txt")


In [ ]:
lookup = family %>%
  distinct(fid) %>%
  mutate(sample = strsplit(fid, "_")) %>%
  unnest(sample)
ukb_smoking$V1 = as.character(ukb_smoking$V1)
ukb_smoking = ukb_smoking %>%
  left_join(lookup, by = c("V1" = "sample"))

aou_smoking$smoking = ifelse(aou_smoking$smoking == '100 Cigs Lifetime: Yes', "Yes", aou_smoking$smoking)
aou_smoking$smoking = ifelse(aou_smoking$smoking == '100 Cigs Lifetime: No', "No", aou_smoking$smoking)
aou_smoking$smoking = ifelse(aou_smoking$smoking %in% c('PMI: Prefer Not To Answer',
                                                        'PMI: Dont Know',
                                                        'PMI: Skip'), NA, aou_smoking$smoking)

In [ ]:
counts$father_smoking = NA
counts$mother_smoking = NA
counts$mother_id = NA
counts$father_id = NA

for (i in 1:nrow(counts)){
    if (i %% 500 == 0){
        message(i)
    }
    if (counts$dataset[i] == "ukb"){
        ukb_sub = ukb_smoking %>% filter(V1 == counts$idv[i])
        counts$father_smoking[i] = ukb_sub$Paternal[1]
        counts$mother_smoking[i] = ukb_sub$Maternal[1]
        counts$mother_id[i] = ukb_sub$fid[1]
        counts$father_id[i] = ukb_sub$fid[1]
    } else {
        trio_sub = aou_trios %>% filter(off == counts$idv[i])
        if (nrow(trio_sub) != 0){
            if (trio_sub$father_id[1] %in% aou_smoking$idv){
                counts$father_smoking[i] = aou_smoking$smoking[which(aou_smoking$idv == trio_sub$father_id[1])]
                counts$father_id[i] = trio_sub$father_id[1]
            }
            if (trio_sub$mother_id[1] %in% aou_smoking$idv){
                counts$mother_smoking[i] = aou_smoking$smoking[which(aou_smoking$idv == trio_sub$mother_id[1])]
                counts$mother_id[i] = trio_sub$mother_id[1]
            }
        }
    }
}
counts_clean = counts %>% filter(!is.na(father_smoking) & !is.na(mother_smoking))
nrow(counts_clean)

In [ ]:
genome = fread("./hg38.chrom.sizes")
genome = genome %>% filter(V1 %in% paste("chr", 1:22, sep = ""))
genome = sum(genome$V2)

In [ ]:
counts_clean = counts_clean %>%
  mutate(
    smoking = factor(
      case_when(
        father_smoking == "Yes" & mother_smoking == "Yes" ~ "both",
        father_smoking == "Yes" & mother_smoking == "No"  ~ "father_only",
        father_smoking == "No"  & mother_smoking == "Yes" ~ "mother_only",
        TRUE                                               ~ "neither"
      ),
      levels = c("neither", "father_only", "mother_only", "both")
    )
  ) %>% 
select(count, denom, mother_age, father_age, mother_id, father_id, smoking, dataset, father_smoking, mother_smoking) %>%
  group_by(smoking, mother_id, father_id, dataset, father_smoking, mother_smoking) %>%
  summarise(across(c(count, denom, mother_age, father_age), mean),
            .groups = "drop")

In [ ]:
df = counts_clean %>% filter(!is.na(mother_age) & !is.na(father_age)) %>% 
    filter(smoking %in% c("both", "neither"))


df = df %>%
  mutate(
    mut_count = count,
    batch    = factor(dataset),
    callable_genome = denom,
  ) %>%
  filter(callable_genome > 0)

# compare to poisson
fit_qp = glm(
  mut_count ~ smoking + batch + mother_age + father_age + offset(log(callable_genome)),
  data = df, family = quasipoisson(link = "log")
)

newdata = expand.grid(
  smoking = c("neither","both"),
  batch = 'ukb',
  father_age = mean(df$father_age),
    mother_age = mean(df$mother_age),
  callable_genome = 1
)

# predict on response scale with se
preds = predict(fit_qp, newdata = newdata, type = "link", se.fit = TRUE)

# ci on log scale
newdata$lower_link = preds$fit - 1.96 * preds$se.fit
newdata$upper_link = preds$fit + 1.96 * preds$se.fit

# transform to response scale
newdata$expected = exp(preds$fit)
newdata$lower    = exp(newdata$lower_link)
newdata$upper    = exp(newdata$upper_link)

# plot
p_rate_cluster_trios = ggplot(newdata, aes(x = smoking, y = expected)) +
  geom_point(size = 3) +
  geom_errorbar(aes(ymin = lower, ymax = upper), width = 0.2) + 
 scale_x_discrete(labels = c("FALSE" = "No", "TRUE" = "Yes")) +
  labs(
    x = "",
    y = "DNM rate",
    title = "DNM rate adjusted for parental age based on trio model",
  ) + 
  theme_bw(base_size = 23) + theme(axis.text.x = element_text(angle = 45, hjust = 1))
mean(df$father_age)
mean(df$mother_age)
summary(fit_qp)
drop1(fit_qp, test = 'F')

In [ ]:
all_96 = fread("./aou_ukb_families_COSMIC_96_all_samples.txt") %>% filter(source == "trio")
# add smoking status 
all_96$father_smoking = NA
all_96$mother_smoking = NA
for (i in 1:nrow(all_96)){
    if (i %% 500 == 0){
        message(i)
    }
    off = strsplit(all_96$fid[i], "_")[[1]][1]
    sub_counts = counts_clean %>% filter(idv == off)
    if (nrow(sub_counts) != 0){
        all_96$father_smoking[i] = sub_counts$father_smoking[1]
        all_96$mother_smoking[i] = sub_counts$mother_smoking[1]
    }
}
all_96_clean = all_96 %>% filter(!is.na(father_smoking) & !is.na(mother_smoking))
nrow(all_96_clean)
cosmic_96 = names(all_96)[2:97]
all_96_clean$smoking = ifelse(all_96_clean$father_smoking == "Yes" & all_96_clean$mother_smoking == "Yes", "both", NA)
all_96_clean$smoking = ifelse(all_96_clean$father_smoking == "No" & all_96_clean$mother_smoking == "No", "neither", all_96_clean$smoking)
all_96_clean = all_96_clean %>% filter(!is.na(smoking))
all_96_smoking = all_96_clean %>% filter(smoking == "both")
all_96_notsmoking = all_96_clean %>% filter(smoking == "neither")
cosmic_96_smoking = colSums(all_96_smoking[,..cosmic_96])
cosmic_96_notsmoking = colSums(all_96_notsmoking[,..cosmic_96])
cosmic_96_smoking = data.frame(context = cosmic_96, count = unname(cosmic_96_smoking), percent = unname(cosmic_96_smoking/sum(cosmic_96_smoking)))
cosmic_96_notsmoking = data.frame(context = cosmic_96, count = unname(cosmic_96_notsmoking), percent = unname(cosmic_96_notsmoking/sum(cosmic_96_notsmoking)))
fwrite(cosmic_96_smoking, "cosmic_96_smoking.txt", sep = "\t", row.names = FALSE, quote = FALSE)
fwrite(cosmic_96_notsmoking, "cosmic_96_notsmoking.txt", sep = "\t", row.names = FALSE, quote = FALSE)